# ToysAD-8K DS0 Proposal and Mask Evaluation

**File:** `06-toysad8k-ds0-mask-evaluation.ipynb`

This evaluator compares two SAM 2.1 routes against the supplied ToysAD-8K masks. The first route uses boxes derived from the supplied masks. The second route uses frozen LocateAnything proposals.

The notebook measures segmentation quality and separates proposal misses from SAM mask errors. It does not calculate B0, B1 or P1. Those ranking experiments use the verified mask routes in the next stage.

## Evaluation contract

- 20 frozen DS0 scenes
- 20 views per scene
- 400 images
- 1,640 supplied reference masks
- 1,640 reference-box SAM masks
- 1,616 automatic-box SAM masks
- mask IoU, Dice and boundary F1 at a three-pixel tolerance
- scene-level summaries and paired asset-cluster bootstrap intervals
- 10,000 bootstrap repetitions with seed 32040
- separate safe, private and model-input outputs

All labels and object associations remain inside the evaluator. The package created for the ranking stage contains masks and route-local identifiers only.

In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import time

from IPython.display import HTML, Image as NotebookImage, display


NOTEBOOK_NAME = "06-toysad8k-ds0-mask-evaluation.ipynb"
OUTPUT_ROOT = Path("/kaggle/working/toysad8k_ds0_proposal_mask_evaluation")
EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/input").is_dir() else "local"

## Locate the frozen inputs and prepare the runtime

In [ ]:
def locate_directory(marker: str) -> Path:
    marker_path = Path(marker)
    candidates: list[Path] = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in input_root.rglob(marker_path.name):
            if path.parts[-len(marker_path.parts) :] == marker_path.parts:
                candidates.append(path.parents[len(marker_path.parts) - 1])
    candidates.extend((Path.cwd(), *Path.cwd().parents))
    for candidate in candidates:
        if (candidate / marker).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate an input containing {marker}")


CODE_ROOT = locate_directory("src/visual_peer_inspector/workflows/mask_evaluation_ds0.py")
MODEL_INPUT_ROOT = locate_directory("neutral_corpus_manifest.jsonl")
EVALUATOR_ROOT = locate_directory("sam_crosswalk/sam_evaluator_crosswalk.jsonl")
PROPOSAL_OUTPUT_ROOT = locate_directory("proposal_results.jsonl")
SAM_OUTPUT_ROOT = locate_directory("private/mask_results.jsonl")

print(json.dumps({"stage": "runtime_install_started"}), flush=True)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(CODE_ROOT / "requirements/mask-evaluation-lock.txt"),
    ],
    timeout=300,
)
print(json.dumps({"stage": "runtime_install_completed"}), flush=True)

sys.path.insert(0, str(CODE_ROOT / "src"))

## Verify provenance and input coverage

In [ ]:
from visual_peer_inspector.data.model_safe_package import file_sha256
from visual_peer_inspector.verification import verify_code_snapshot_record


snapshot = verify_code_snapshot_record(CODE_ROOT, CODE_ROOT / "snapshot_record.json")
notebook_path = CODE_ROOT / "notebooks" / NOTEBOOK_NAME
if not notebook_path.is_file():
    raise FileNotFoundError(f"The code snapshot does not contain {NOTEBOOK_NAME}")

evaluator_record = json.loads(
    (EVALUATOR_ROOT / "package_record.json").read_text(encoding="utf-8")
)
model_input_record = json.loads(
    (MODEL_INPUT_ROOT / "package_record.json").read_text(encoding="utf-8")
)
proposal_record = json.loads(
    (PROPOSAL_OUTPUT_ROOT / "report/run_record.json").read_text(encoding="utf-8")
)
sam_record = json.loads(
    (SAM_OUTPUT_ROOT / "safe/run_record.json").read_text(encoding="utf-8")
)

if proposal_record.get("status") != "completed":
    raise ValueError("Notebook 04 proposal evidence is incomplete")
if sam_record.get("status") != "completed":
    raise ValueError("Notebook 05 SAM evidence is incomplete")
if sam_record.get("producer_kind") != "sam2.1_model":
    raise ValueError("Notebook 05 did not use the pinned SAM 2.1 model")
if sam_record.get("inference_stage") != "full":
    raise ValueError("Notebook 05 did not complete the full DS0 run")
if model_input_record.get("image_count") != 400:
    raise ValueError("The qualitative RGB package does not contain 400 images")

platform_versions = snapshot.get("platform_versions", {})
if "kaggle_notebook_version_06" not in platform_versions:
    raise ValueError("The code snapshot does not pin Notebook 06")
if "code_dataset_version" not in platform_versions:
    raise ValueError("The code snapshot does not pin its Kaggle dataset version")

evaluator_provenance = {
    "git_commit": snapshot["git_commit"],
    "git_dirty": snapshot["git_dirty"],
    "snapshot_sha256": snapshot["snapshot_sha256"],
    "precommit_candidate": snapshot["precommit_candidate"],
    "notebook_sha256": file_sha256(notebook_path),
    "kaggle_notebook_slug": (
        "esreekarreddy/toysad-8k-ds0-proposal-and-mask-evaluation"
    ),
    "kaggle_notebook_version": str(platform_versions["kaggle_notebook_version_06"]),
    "code_dataset_version": str(platform_versions["code_dataset_version"]),
}

input_summary = {
    "code_files": snapshot["file_count"],
    "code_snapshot_sha256": snapshot["snapshot_sha256"],
    "git_commit": snapshot["git_commit"],
    "precommit_candidate": snapshot["precommit_candidate"],
    "evaluator_scenes": evaluator_record["scene_count"],
    "evaluator_images": evaluator_record["image_count"],
    "verified_rgb_images": model_input_record["image_count"],
    "supplied_masks": evaluator_record["supplied_mask_count"],
    "proposal_status": proposal_record["status"],
    "sam_status": sam_record["status"],
    "sam_masks": sam_record["summary"]["mask_count"],
    "sam_repeat_agreement": sam_record["repeat_agreement"]["mask_digest_agreement_rate"],
    "python_version": platform.python_version(),
    "numpy_version": importlib.metadata.version("numpy"),
    "pandas_version": importlib.metadata.version("pandas"),
    "pillow_version": importlib.metadata.version("Pillow"),
    "scipy_version": importlib.metadata.version("scipy"),
}
display(input_summary)

## Run the complete evaluator

In [ ]:
from visual_peer_inspector.workflows.mask_evaluation_ds0 import (
    MODEL_EVIDENCE_MODE,
    run_ds0_mask_evaluation,
)


evaluation_started = time.perf_counter()
print(json.dumps({"stage": "mask_evaluation_started"}), flush=True)
run_record = run_ds0_mask_evaluation(
    evaluator_root=EVALUATOR_ROOT,
    model_input_root=MODEL_INPUT_ROOT,
    proposal_output_root=PROPOSAL_OUTPUT_ROOT,
    sam_output_root=SAM_OUTPUT_ROOT,
    output_root=OUTPUT_ROOT,
    evaluator_provenance=evaluator_provenance,
    evaluation_mode=MODEL_EVIDENCE_MODE,
    mask_iou_threshold=0.75,
    boundary_tolerance_pixels=3,
    bootstrap_repetitions=10_000,
    bootstrap_seed=32_040,
)
evaluation_seconds = round(time.perf_counter() - evaluation_started, 3)
print(
    json.dumps(
        {
            "stage": "mask_evaluation_completed",
            "elapsed_seconds": evaluation_seconds,
            "scientific_gates_passed": run_record["scientific_gates_passed"],
        },
        sort_keys=True,
    ),
    flush=True,
)

if not run_record["completion_gate_passed"]:
    raise RuntimeError("Notebook 06 did not complete its evaluation contract")
display(run_record["candidate_decision"])

## Aggregate results

In [ ]:
import pandas as pd


safe_root = OUTPUT_ROOT / "safe"
completion_gates = pd.read_csv(safe_root / "completion_gates.csv")
decision_gates = pd.read_csv(safe_root / "decision_gates.csv")
proposal_summary = pd.read_csv(safe_root / "proposal_gate_summary.csv")
route_summary = pd.read_csv(safe_root / "route_summary.csv")
view_route_summary = pd.read_csv(safe_root / "view_route_summary.csv")
route_comparison = pd.read_csv(safe_root / "route_comparison.csv")
bootstrap_intervals = pd.read_csv(safe_root / "bootstrap_intervals.csv")
key_results = route_summary[[
    "route",
    "reference_count",
    "prediction_count",
    "detection_aware_recall_at_0_75",
    "mean_matched_iou",
    "mean_matched_boundary_f1",
]].copy()
key_results = key_results.rename(columns={
    "route": "Mask route",
    "reference_count": "Reference objects",
    "prediction_count": "Predicted masks",
    "detection_aware_recall_at_0_75": "Recall at IoU 0.75",
    "mean_matched_iou": "Mean matched IoU",
    "mean_matched_boundary_f1": "Mean boundary F1",
})
metric_columns = ["Recall at IoU 0.75", "Mean matched IoU", "Mean boundary F1"]
key_results[metric_columns] = key_results[metric_columns].round(4)

display(HTML("<h4>Key mask results</h4>"))
display(key_results)
display(HTML("<h4>Completion gates</h4>"))
display(completion_gates)
display(HTML("<h4>Decision gates</h4>"))
display(decision_gates)
display(HTML("<h4>Proposal gate summary</h4>"))
display(proposal_summary)
display(HTML("<h4>Mask route summary</h4>"))
display(route_summary)
display(HTML("<h4>Per-view route summary</h4>"))
display(view_route_summary)
display(HTML("<h4>Paired route comparison</h4>"))
display(route_comparison)
display(HTML("<h4>Paired asset-cluster bootstrap intervals</h4>"))
display(bootstrap_intervals)

In [ ]:
for chart_name in (
    "reference_box_mask_iou_distribution.png",
    "route_metric_comparison.png",
    "automatic_box_mask_iou_distribution.png",
    "reference_box_boundary_f1_distribution.png",
    "automatic_box_boundary_f1_distribution.png",
    "reference_box_recall_by_view.png",
    "automatic_box_prompt_iou_vs_mask_iou.png",
    "automatic_box_recall_by_view.png",
    "misses_and_extras_by_route.png",
):
    display(NotebookImage(filename=str(safe_root / chart_name)))

## Qualitative mask evaluation

The first panel contains two deterministic examples from the strong, typical and challenging automatic-route IoU ranks. Each case shows the RGB view, supplied mask, automatic SAM mask and a direct agreement map. Green means reference only, red means prediction only and blue means overlap.

The second panel retains the twelve lowest-IoU matched masks as failure evidence. These private panels support interpretation and report figures. They are not uploaded to ClearML or copied into the public repository outputs.

In [ ]:
private_root = OUTPUT_ROOT / "private"
for image_name in (
    "representative_automatic_mask_cases.png",
    "worst_automatic_mask_pairs.png",
):
    image_path = private_root / image_name
    if not image_path.is_file():
        raise FileNotFoundError(f"Missing qualitative evidence: {image_name}")
    display(NotebookImage(filename=str(image_path)))

## Verify output separation

In [ ]:
from visual_peer_inspector.data.mask_route_package import verify_mask_route_package
from visual_peer_inspector.workflows.mask_evaluation_ds0 import verify_artifact_manifest


safe_manifest = verify_artifact_manifest(OUTPUT_ROOT / "safe")
private_manifest = verify_artifact_manifest(OUTPUT_ROOT / "private")
model_input_record = verify_mask_route_package(OUTPUT_ROOT / "model_input")

expected_route_counts = {
    "supplied_mask": 1640,
    "reference_box": 1640,
    "automatic_box": 1616,
}
if model_input_record.get("route_counts") != expected_route_counts:
    raise RuntimeError("The model-facing mask routes are incomplete")

display(
    {
        "safe_artifacts": safe_manifest["artifact_count"],
        "private_artifacts": private_manifest["artifact_count"],
        "model_input_checks_passed": model_input_record["all_checks_passed"],
        "route_counts": model_input_record["route_counts"],
    }
)

## Record aggregate evidence in ClearML

Tracking starts after evaluation and output verification. The finalizer accepts the aggregate tables, charts and hashes under `safe` only. It rejects masks, boxes, images, identifiers, crosswalks and private failure records.

In [ ]:
def prepare_tracking_environment() -> dict[str, str]:
    environment = os.environ.copy()
    existing_pythonpath = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = str(CODE_ROOT / "src") + (
        os.pathsep + existing_pythonpath if existing_pythonpath else ""
    )
    if EXECUTION_ENVIRONMENT == "kaggle":
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
            value = client.get_secret(name)
            if not value:
                raise RuntimeError(f"Kaggle Secret {name} is missing or disabled")
            environment[name] = value
    environment.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
    environment.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml")
    environment.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
    environment.setdefault("CLEARML_LOG_ENVIRONMENT", "")
    environment.setdefault("VPI_CLEARML_REQUEST_TIMEOUT_SECONDS", "30")
    return environment


tracking_receipt = OUTPUT_ROOT / "tracking_receipt.json"
tracking_command = [
    sys.executable,
    str(CODE_ROOT / "scripts/finalize_mask_evaluation_clearml.py"),
    "--safe-root",
    str(safe_root),
    "--receipt",
    str(tracking_receipt),
    "--mode",
    "online",
    "--wall-time-seconds",
    "180",
]

try:
    completed_tracking = subprocess.run(
        tracking_command,
        env=prepare_tracking_environment(),
        text=True,
        capture_output=True,
        timeout=210,
        check=True,
    )
    tracking_result = json.loads(tracking_receipt.read_text(encoding="utf-8"))
except Exception as tracking_error:
    tracking_result = {
        "schema_version": 1,
        "status": "failed",
        "tracking_mode": "online",
        "clearml_task_id": None,
        "error_type": type(tracking_error).__name__,
    }
    tracking_receipt.write_text(
        json.dumps(tracking_result, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

display(tracking_result)

## Final result

In [ ]:
routes = {row["route"]: row for row in run_record["route_summary"]}
final_summary = {
    "status": run_record["status"],
    "scientific_gates_passed": run_record["scientific_gates_passed"],
    "reference_recall_at_0_75": routes["reference_box"]["detection_aware_recall_at_0_75"],
    "reference_mean_matched_iou": routes["reference_box"]["mean_matched_iou"],
    "automatic_recall_at_0_75": routes["automatic_box"]["detection_aware_recall_at_0_75"],
    "automatic_mean_matched_iou": routes["automatic_box"]["mean_matched_iou"],
    "proceed_to_ds1": run_record["candidate_decision"]["proceed_to_ds1"],
    "final_route_acceptance": run_record["candidate_decision"]["final_route_acceptance"],
    "ranking_metrics_status": run_record["ranking_metrics_status"],
    "evaluation_seconds": evaluation_seconds,
    "tracking_status": tracking_result["status"],
    "clearml_task_id": tracking_result.get("clearml_task_id"),
}
display(final_summary)

if not run_record["scientific_gates_passed"]:
    raise RuntimeError("The DS0 proposal and mask gate requires review")

The DS0 result is a provisional integration decision. Passing this notebook supports the 100-scene screen. It does not accept the automatic route as the final web application default. B0, B1 and P1 remain part of the ranking stage.